# BEV-Track on a free Colab GPU

Runs the GPU half of BEV-Track on Colab's free T4. Everything here is free: Colab, the nuScenes-mini
download, the pretrained checkpoint, and Google Drive for caching.

**Before you start**
1. *Runtime → Change runtime type → T4 GPU.*
2. Put `bev-track.zip` (from `scripts/make_colab_bundle.sh`) in Google Drive at `MyDrive/bevtrack/`,
   or set `GIT_URL` below if the repo is on GitHub.
3. CAN bus data (BEVFormer needs it; free but needs a login): sign in at
   [nuscenes.org/nuscenes#download](https://www.nuscenes.org/nuscenes#download), then either
   * copy the *CAN bus expansion* download link into `CAN_BUS_URL` below (signed links expire; use it right away), or
   * upload `can_bus.zip` to `MyDrive/bevtrack/`.

**Time:** first session about 1.5–2 h (the environment build is ~25 min of that). Later sessions reuse the
compiled mmdet3d wheel from Drive. Free Colab can disconnect when idle, so keep the tab open.
Each step writes to Drive, so a disconnect loses at most one step.

In [ ]:
GIT_URL = 'https://github.com/abhijnya4601/bev-track.git'   # empty -> use MyDrive/bevtrack/bev-track.zip
CAN_BUS_URL = ''    # paste the signed can_bus.zip link here in Colab (don't commit it); empty -> MyDrive/bevtrack/can_bus.zip

from google.colab import drive
drive.mount('/content/drive')
import os, subprocess
DRIVE = '/content/drive/MyDrive/bevtrack'
os.makedirs(DRIVE, exist_ok=True)
os.environ.update(DRIVE=DRIVE, GIT_URL=GIT_URL, CAN_BUS_URL=CAN_BUS_URL)
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], capture_output=True, text=True).stdout)

## 1. Get the code

In [ ]:
%%bash
set -e
cd /content
rm -rf bev-track
if [ -n "$GIT_URL" ]; then
  git clone --recurse-submodules "$GIT_URL" bev-track
else
  unzip -q "$DRIVE/bev-track.zip" -d /content
fi
ls bev-track

## 2. Build the pinned environment (Python 3.8, torch 1.9.1+cu111, mmcv-full 1.4.0, mmdet3d 0.17.1)

Colab ships Python 3.12 and CUDA 12, far newer than BEVFormer supports, so this builds a separate
environment with micromamba. torch 1.9's cu111 build runs on Colab's newer driver. mmdet3d's CUDA ops
are compiled with nvcc 11.8 (same CUDA major version, which is all torch 1.9 checks) and gcc 9
(Colab's gcc 11 is too new for torch 1.9's headers). The compiled wheel is cached in Drive.

In [ ]:
%%bash
set -e
export ENV=/content/bevenv MAMBA_ROOT_PREFIX=/content/mamba
cd /content
if [ ! -x bin/micromamba ]; then curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj bin/micromamba; fi
MM=/content/bin/micromamba

$MM create -y -q -p $ENV -c conda-forge python=3.8
# CUDA 11 runtime library (small); the compiled mmdet3d ops link against libcudart.so.11.0
$MM install -y -q -p $ENV -c nvidia/label/cuda-11.8.0 cuda-cudart
PIP="$ENV/bin/python -m pip install -q"
cat > /content/constraints.txt <<EOF
numpy==1.19.5
setuptools==59.5.0
yapf==0.40.1
pillow==9.5.0
EOF
C="-c /content/constraints.txt"

$PIP --upgrade "pip<24"
$PIP $C setuptools==59.5.0 wheel ninja pillow==9.5.0
# Pillow 10 removed Image.LINEAR, which detectron2 0.6 (imported by BEVFormer's plugin) uses at import time
$PIP $C torch==1.9.1+cu111 torchvision==0.10.1+cu111 torchaudio==0.9.1 -f https://download.pytorch.org/whl/torch_stable.html
$PIP $C mmcv-full==1.4.0 -f https://download.openmmlab.com/mmcv/dist/cu111/torch1.9.0/index.html
$PIP $C mmdet==2.14.0 mmsegmentation==0.14.1
# BEVFormer docs/install.md pins (ipython/pylint dev tools omitted)
$PIP $C einops fvcore seaborn iopath==0.1.9 timm==0.6.13 typing-extensions==4.5.0 numpy==1.19.5 \
    matplotlib==3.5.2 numba==0.48.0 pandas==1.4.4 scikit-image==0.19.3 yapf==0.40.1
# mmdet3d v0.17.1 runtime requirements, pinned so nothing drags numpy forward
$PIP $C lyft_dataset_sdk==0.0.8 networkx==2.2 nuscenes-devkit==1.1.9 plyfile tensorboard trimesh==2.35.39 pyquaternion scipy
$PIP $C detectron2 -f https://dl.fbaipublicfiles.com/detectron2/wheels/cu111/torch1.9/index.html
$PIP $C pytest

WHEEL=$(ls $DRIVE/wheels/mmdet3d-0.17.1*.whl 2>/dev/null | head -1 || true)
if [ -z "$WHEEL" ]; then
  echo ">>> Building mmdet3d 0.17.1 CUDA ops (~20 min, cached to Drive afterwards)"
  apt-get -qq install -y gcc-9 g++-9 > /dev/null
  $MM install -y -q -p $ENV -c nvidia/label/cuda-11.8.0 cuda-nvcc cuda-cudart-dev cuda-libraries-dev cuda-cccl
  ln -sfn $ENV/lib $ENV/lib64
  rm -rf /content/mmdetection3d
  git clone -q --branch v0.17.1 --depth 1 https://github.com/open-mmlab/mmdetection3d.git /content/mmdetection3d
  cd /content/mmdetection3d
  PATH=$ENV/bin:$PATH CUDA_HOME=$ENV CC=gcc-9 CXX=g++-9 FORCE_CUDA=1 TORCH_CUDA_ARCH_LIST="7.5" MAX_JOBS=4 \
    $ENV/bin/python -m pip wheel . --no-deps --no-build-isolation -w $DRIVE/wheels
  WHEEL=$(ls $DRIVE/wheels/mmdet3d-0.17.1*.whl | head -1)
fi
$PIP --no-deps "$WHEEL"

# Runner used by every later cell
cat > /content/run <<EOF
#!/bin/bash
export LD_LIBRARY_PATH=$ENV/lib:\${LD_LIBRARY_PATH:-} PATH=$ENV/bin:\$PATH
cd /content/bev-track
exec "\$@"
EOF
chmod +x /content/run
/content/run python -c "import torch, mmcv, mmdet, mmdet3d; from mmdet3d.ops import Voxelization; print('torch', torch.__version__, 'cuda ok:', torch.cuda.is_available(), '| mmcv', mmcv.__version__, '| mmdet3d', mmdet3d.__version__)"

## 3. Sanity: BEV-Track's unit tests inside the pinned env (Python 3.8, numpy 1.19)

In [ ]:
!/content/run python -m pytest -q 2>&1 | tail -3

## 4. Data: nuScenes-mini (direct download) + CAN bus

In [ ]:
%%bash
set -e
cd /content/bev-track
mkdir -p data/nuscenes
if [ ! -d data/nuscenes/v1.0-mini ]; then
  wget -q -O /content/v1.0-mini.tgz https://www.nuscenes.org/data/v1.0-mini.tgz
  tar -xzf /content/v1.0-mini.tgz -C data/nuscenes && rm /content/v1.0-mini.tgz
fi
if [ ! -d data/can_bus ]; then
  if [ -n "$CAN_BUS_URL" ]; then wget -q -O /content/can_bus.zip "$CAN_BUS_URL"; else cp "$DRIVE/can_bus.zip" /content/can_bus.zip; fi
  unzip -q /content/can_bus.zip -d data && rm /content/can_bus.zip
fi
ls data data/nuscenes data/can_bus | head -20

## 5. Info files, GT files, split report, pretrained checkpoint

In [ ]:
!/content/run bash scripts/gpu_pipeline.sh 0 1 2 3 4

## 6. Baseline: unmodified pretrained checkpoint (10 classes mapped to 5) on the 4 clean scenes

In [ ]:
!/content/run bash scripts/gpu_pipeline.sh 5
!mkdir -p $DRIVE/results && cp -r /content/bev-track/results/* $DRIVE/results/

## 7. Fine-tune the 5-class head (~45 min on a T4)

Two dataloader workers instead of four, because free Colab has ~12 GB RAM.

In [ ]:
!TRAIN_CFG_OPTIONS="data.workers_per_gpu=2" /content/run bash scripts/gpu_pipeline.sh 6
!mkdir -p $DRIVE/work_dirs && cp /content/bev-track/work_dirs/bevformer_tiny_5cls/latest.pth $DRIVE/work_dirs/ && cp /content/bev-track/work_dirs/bevformer_tiny_5cls/*.log $DRIVE/work_dirs/

## 8. Evaluate the fine-tuned model, extract failure cases, run tracking

In [ ]:
!/content/run bash scripts/gpu_pipeline.sh 7 8
!cp -r /content/bev-track/results/* $DRIVE/results/

## 9. Results

In [ ]:
import pandas as pd
import os
for name in ['pretrained', 'finetuned']:
    f = f'/content/bev-track/results/{name}/metrics_by_class.csv'
    if not os.path.exists(f):
        print(f'--- {name}: no results yet (did its step fail above?) ---'); continue
    df = pd.read_csv(f'/content/bev-track/results/{name}/metrics_by_class.csv')
    print(f'--- {name} ---'); print(df[['class', 'n_gt', 'n_pred', 'AP', 'NDS', 'ATE', 'ASE', 'AOE', 'AVE']].to_string(index=False))
if os.path.exists('/content/bev-track/results/finetuned/metrics_by_distance.csv'):
    print(pd.read_csv('/content/bev-track/results/finetuned/metrics_by_distance.csv')[['slice', 'class', 'n_gt', 'AP']].to_string(index=False))
if os.path.exists('/content/bev-track/results/tracking_metrics.csv'):
    print(pd.read_csv('/content/bev-track/results/tracking_metrics.csv').to_string(index=False))
print('Everything is also in', DRIVE, '-> copy results/ back into the repo on your laptop.')